# 03_Statistical_Analysis: Hypothesis Testing and Segmentation

This notebook performs statistical analyses to identify meaningful relationships between transaction attributes and suspicious activity, supporting data-driven risk insights.


## Setup and Load Data

Import the cleaned transaction data and necessary statistical libraries.

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set(style='whitegrid')

DATA_PROCESSED_DIR = Path('data/processed')
df = pd.read_csv(DATA_PROCESSED_DIR / 'cleaned_transactions.csv')
print('Data loaded:', df.shape)


## Correlation Analysis for Numeric Fields

Compute the correlation matrix for numeric variables and visualize it.

In [ ]:

numeric_cols = df.select_dtypes(include=[np.number]).columns
corr = df[numeric_cols].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix of Numeric Features')
plt.show()


## Chi-square Tests for Categorical Variables

Perform chi-square tests between categorical predictors and the target variable `Is_laundering`.

In [ ]:

from scipy.stats import chi2_contingency

categorical_cols = df.select_dtypes(include=['object','category']).columns
for col in categorical_cols:
    if col != 'is_laundering':
        contingency = pd.crosstab(df[col], df['is_laundering'])
        if contingency.shape[0] > 1 and contingency.shape[1] > 1:
            chi2, p, dof, expected = chi2_contingency(contingency)
            print(f'Chi-square test for {col}: p-value = {p:.5f}')


## T-test or Mann-Whitney Test on Transaction Amount

Compare transaction amounts between suspicious and non-suspicious groups to see if there is a significant difference.

In [ ]:

# Use Mann-Whitney U test when distributions are non-normal
if 'amount' in df.columns and 'is_laundering' in df.columns:
    suspicious = df[df['is_laundering']==1]['amount']
    normal = df[df['is_laundering']==0]['amount']
    u_stat, p_val = stats.mannwhitneyu(suspicious, normal, alternative='two-sided')
    print('Mann-Whitney U statistic:', u_stat, 'p-value:', p_val)


## Account-level Segmentation and Risk Bands

Segment sender accounts by the rate of suspicious transactions and assign risk bands: Low, Medium, High.

In [ ]:

if {'sender_account','is_laundering'}.issubset(df.columns):
    sender_stats = df.groupby('sender_account')['is_laundering'].mean()
    # Define quantiles for risk bands
    quantiles = sender_stats.quantile([0.33, 0.66])
    def assign_band(x):
        if x <= quantiles.iloc[0]:
            return 'Low'
        elif x <= quantiles.iloc[1]:
            return 'Medium'
        else:
            return 'High'
    sender_risk_band = sender_stats.apply(assign_band)
    # Display counts per risk band
    print(sender_risk_band.value_counts())


## Business Interpretation

Interpret the statistical results and explain how they inform transaction risk monitoring strategies.

In [ ]:

print('Interpretation:
')
print('- Strong correlations or significant p-values indicate relationships between features and suspicious activity.')
print('- High-risk accounts identified here can be prioritized for enhanced monitoring.')
